# 📓 L6 Colab 2 — LLM Storytelling (Olist)

Usa il LLM (Ollama llama3.2:3b) per generare narrative di business a partire
dai KPI e dagli insight calcolati in Colab 1.

**Input:** `L6_out/bi/` — kpis.json + orders_wide.csv (prodotti da Colab 1)
**Output:** `L6_out/storytelling.json` — narrative generate dal LLM

In [1]:
import pandas as pd
import json, os, requests
from pathlib import Path

# ── Config LLM ────────────────────────────────────────────────────
LLM_PROVIDER   = 'ollama'          # 'ollama' | 'openai' | 'anthropic'
OLLAMA_MODEL   = 'llama3.2:3b'
OLLAMA_URL     = 'http://localhost:11434/api/chat'
OPENAI_MODEL   = 'gpt-4o-mini'
ANTHROPIC_MODEL= 'claude-haiku-4-5-20251001'
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

# ── Percorsi ──────────────────────────────────────────────────────
BI_PATH  = Path('../2_scripts/L6_out/bi/')
OUT_PATH = Path('../2_scripts/L6_out/')

print(f'Provider: {LLM_PROVIDER}')

Provider: ollama


In [2]:
def call_llm(system_prompt, user_prompt, max_tokens=800):
    try:
        if LLM_PROVIDER == 'ollama':
            r = requests.post(OLLAMA_URL, json={
                'model': OLLAMA_MODEL, 'stream': False,
                'messages': [{'role':'system','content':system_prompt},
                             {'role':'user',  'content':user_prompt}]
            }, timeout=120)
            r.raise_for_status()
            return r.json()['message']['content']
        elif LLM_PROVIDER == 'openai':
            import openai
            client = openai.OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{'role':'system','content':system_prompt},
                           {'role':'user','content':user_prompt}])
            return resp.choices[0].message.content
        elif LLM_PROVIDER == 'anthropic':
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            msg = client.messages.create(
                model=ANTHROPIC_MODEL, max_tokens=max_tokens,
                system=system_prompt,
                messages=[{'role':'user','content':user_prompt}])
            return msg.content[0].text
    except Exception as e:
        return f'[LLM ERROR] {e}'

print('call_llm pronta.')

call_llm pronta.


In [3]:
# ── Carica KPI e dati aggregati da Colab 1 ───────────────────────
with open(BI_PATH / 'kpis.json') as f:
    kpis = json.load(f)

wide = pd.read_csv(BI_PATH / 'orders_wide.csv')

# Aggregati compatti da passare al LLM (riduce i token)
state_rev = (
    wide.groupby('customer_state')['revenue']
        .sum().sort_values(ascending=False)
        .head(5).round(0).to_dict()
)
cat_rev = (
    wide.groupby('product_category_name_english')['revenue']
        .sum().sort_values(ascending=False)
        .head(5).round(0).to_dict()
)
year_rev = (
    wide.groupby('year')['revenue']
        .sum().round(0).to_dict()
)
state_delay = (
    wide.groupby('customer_state')['delivery_delay_days']
        .mean().sort_values(ascending=False)
        .head(5).round(1).to_dict()
)

print('Dati caricati.')
print('KPI:', kpis)

Dati caricati.
KPI: {'total_revenue': 15200441.25, 'total_sales': 112650, 'unique_orders': 98666, 'unique_customers': 98666, 'avg_price': 115.3, 'avg_review': 4.04, 'avg_delay_days': -12.03, 'top_state': 'SP', 'top_category': 'health_beauty', 'top_payment': 'credit_card'}


## UC1 — Generazione automatica di insight

Il LLM analizza i KPI e i dati aggregati e produce insight nel formato
**WHAT / WHY / SO-WHAT** usato nel corso di storytelling.

In [4]:
SYSTEM_STORY = (
    'Sei un senior Business Analyst specializzato in e-commerce. '
    'Analizzi i dati di Olist, marketplace brasiliano (2016-2018). '
    'Produci insight nel formato: WHAT (cosa osservi), WHY (perche), '
    'SO-WHAT (implicazione per il business). '
    'Scrivi in italiano. Sii conciso e orientato al business.'
)

user_prompt = (
    f'Ecco i KPI principali del Data Warehouse Olist:\n'
    f'{json.dumps(kpis, indent=2, ensure_ascii=False)}\n\n'
    f'Revenue per anno: {json.dumps(year_rev)}\n'
    f'Top 5 stati per revenue: {json.dumps(state_rev)}\n'
    f'Top 5 categorie per revenue: {json.dumps(cat_rev)}\n'
    f'Top 5 stati per ritardo medio (giorni): {json.dumps(state_delay)}\n\n'
    f'Genera 3 insight rilevanti per il business nel formato WHAT/WHY/SO-WHAT.'
)

insight_auto = call_llm(SYSTEM_STORY, user_prompt)
print('=== INSIGHT AUTOMATICI ===')
print(insight_auto)

=== INSIGHT AUTOMATICI ===
Ecco tre insight rilevanti per il business Olist:

**Insight 1:**
WHAT: Il stato di São Paulo è il principale contributore al revenue, seguito da quello di Rio de Janeiro e Minas Gerais.
WHY: La concentrazione del revenue in pochi stati suggerisce una forte presenza di vendite localizzate, che potrebbero essere opportunità per migliorare la soddisfazione dei clienti locali.
SO-WHAT: Questo insieme di dati suggerisce l'importanza di concentrarsi sulle regioni con le maggiore domanda e di offrire soluzioni personalizzate ai clienti locali, in modo da aumentare la fedeltà e la soddisfazione. Ad esempio, potrebbe essere utile creare campagne pubblicitarie mirate alle regioni più promettenti.

**Insight 2:**
WHAT: Il top pagamento è il credit card, seguito dal PayPal e dal deposito bancario.
WHY: La maggior parte delle transazioni viene effettuata tramite credit card, che potrebbe suggerire una scarsa fiducia dei clienti nei pagamenti online.
SO-WHAT: Questo insie

## UC2 — Arricchimento WHY

Parte da un'osservazione statistica (WHAT già noto) e chiede al LLM
di generare ipotesi causali (WHY) nel contesto Olist.

In [5]:
SYSTEM_WHY = (
    'Sei un esperto di e-commerce brasiliano. '
    'Data un osservazione statistica su Olist, genera ipotesi causali '
    'plausibili e azioni raccomandate. Scrivi in italiano.'
)

# WHAT già noto: gli stati del Nord hanno ritardi molto più alti
what_observed = (
    'I clienti degli stati del Nord del Brasile (AM, PA, RR, AP) '
    f'hanno un ritardo medio di consegna di circa {state_delay} giorni, '
    'nettamente superiore alla media nazionale. '
    'Contestualmente, la review media in quegli stati è inferiore alla media.'
)

why_prompt = (
    f'Osservazione (WHAT): {what_observed}\n\n'
    f'Genera 3 ipotesi WHY e 2 azioni SO-WHAT per il management di Olist.'
)

why_resp = call_llm(SYSTEM_WHY, why_prompt)
print('=== WHY HYPOTHESES ===')
print(why_resp)

=== WHY HYPOTHESES ===
**Analysero i dati e generati le ipotesi**

**Ipotesi Causali:**

1. **Mancanza di capacità di gestione dei carichi**: I stati del Nord del Brasile potrebbero avere una crescita rapida dell'e-commerce, ma la capacità di gestione dei carichi dei sistemi e delle infrastrutture non sarebbe stata adeguata, causando ritardi nella consegna degli ordini.
2. **Mancanza di strategia logistica efficace**: La mancanza di una strategia logistica ben definita e implementata potrebbe essere il motivo del ritardo nella consegna degli ordini, poiché gli ordini non vengono trattati in modo prioritario o con la giusta capacità.
3. **Problemi di infrastruttura**: I stati del Nord del Brasile potrebbero avere problemi di infrastruttura logistica, come ad esempio una rete di trasporti limitata o una gestione inefficiente delle spedizioni, che contribuono al ritardo nella consegna degli ordini.

**Azioni SO-WHAT:**

1. **Migliorare la capacità di gestione dei carichi**: Olist dovrebbe

In [6]:
# ── Salva tutti gli output in storytelling.json ───────────────────
story_output = {
    'kpis':            kpis,
    'insight_auto':    insight_auto,
    'why_hypotheses':  why_resp,
}

with open(OUT_PATH / 'storytelling.json', 'w', encoding='utf-8') as f:
    json.dump(story_output, f, indent=2, ensure_ascii=False)

print('Storytelling salvato -> L6_out/storytelling.json')

Storytelling salvato -> L6_out/storytelling.json
